# ⚖️ Synthetic Data EDA & ML Metrics
This notebook explores the highly calibrated supply-chain synthetic dataset to test correlations, leakage limits, and positive $R^2$ verification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

## 1. Load Data
Pulling the final dataset.

In [ ]:
df = pd.read_csv('../data/final_combined_data.csv')
print(f"Loaded Shape: {df.shape}")
df.head()

## 2. Bivariate Check (Return Probability Drivers)
Checking correlations without leakages to guarantee structural learning.
**Delivery Delay vs Return Rate:**

In [ ]:
delay_groups = df.groupby(pd.cut(df['delivery_delay'], bins=[-1, 0, 2, 5, 20]))['is_returned'].mean()
delay_groups.plot(kind='bar', color='salmon', title='Return Rate by Delay in Days')
plt.ylabel('Return Prob')
plt.xticks(rotation=45)
plt.show()

**Category Variations:**

In [ ]:
cat_groups = df.groupby('category')['is_returned'].mean()
cat_groups.plot(kind='bar', color='skyblue', title='Return Rate by Category')
plt.ylabel('Return Prob')
plt.xticks(rotation=45)
plt.show()

## 3. ML Model Evaluator
Validating target thresholds (e.g. $R^2 > 0.3$, $RMSE$, $MAE$)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, accuracy_score, roc_auc_score

# Drop Leakage ID cols
ignore_cols = [
    "order_id", "customer_id", "product_id", "order_date", 
    "return_reason", "return_days_after_delivery", "product_name",
    "city", "state", "pincode", "brand", "warehouse_city", "delivery_city", 
    "courier_partner", "order_day_of_week"
]
X_raw = df.drop(columns=[c for c in ignore_cols if c in df.columns], errors='ignore')
X_raw = X_raw.drop(columns=["is_returned"], errors='ignore')
y = df["is_returned"]

X_encoded = pd.get_dummies(X_raw, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

In [ ]:
reg = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
reg.fit(X_train, y_train)
y_pred_reg = reg.predict(X_test)

r2 = r2_score(y_test, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_reg))
mae = mean_absolute_error(y_test, y_pred_reg)

print(f"Regression Targeting metrics for R-Square Validations:")
print(f"R-Squared: {r2:.4f}")
print(f"RMSE:      {rmse:.4f}")
print(f"MAE:       {mae:.4f}")

In [ ]:
clf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
y_prob = clf.predict_proba(X_test)[:, 1]
y_pred = clf.predict(X_test)

print(f"Classification Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score          : {roc_auc_score(y_test, y_prob):.4f}")